In [ ]:
# === 02_preprocess.ipynb - alignment + feature extraction ===
# Run AFTER 01_setup_data.ipynb has produced metadata.csv and the split.
#
# Settings: same as notebook 01 (GPU T4 x2, Internet On, Persistence Files only)
#
# Order matters: vocab (Telugu) -> source patches -> alignment -> features.
# The patch cell is self-contained and runs BEFORE alignment: it proves the
# audio stack works on this image in seconds instead of failing after hours.
LANG = "Telugu"  # English run complete -> Telugu is now the active language
SSTF = "/kaggle/working/sstf"
EXPECTED_CODE = "2026.11.03.5"

import sys, os, re
os.chdir(SSTF)
sys.path.insert(0, SSTF)

# Which build of the code is actually on disk? (a stale Kaggle dataset is the
# single most common reason a "fixed" pipeline still fails)
_init = f"{SSTF}/common/__init__.py"
if os.path.exists(_init):
    _m = re.search(r'__version__ = "([^"]+)"', open(_init, encoding="utf-8").read())
    _v = _m.group(1) if _m else "unknown"
    print("sstf code version:", _v)
    if _v != EXPECTED_CODE:
        print("WARNING: mounted code is not the current build - re-upload "
              "sstf_code.zip. Cell 3 below patches the upstream clone itself, "
              "so this notebook still works.")


In [ ]:
# 1) make sure the data symlink is in place, then rebuild the manifest/split.
os.makedirs(f"{SSTF}/data", exist_ok=True)
if not os.path.exists(f"{SSTF}/data/{LANG}"):
    os.symlink(f"/kaggle/working/data/{LANG}", f"{SSTF}/data/{LANG}")

!python project/{LANG}/preprocess.py --stage manifest


In [ ]:
# 2) Telugu only: build the grapheme vocabulary (the symbol patch needs it).
#    English uses the reference phoneme set, so this is a no-op there.
if LANG == "Telugu":
    !python project/Telugu/preprocess.py --stage vocab
else:
    print("English: reference phoneme symbols, no vocab stage")


In [ ]:
# 3) Patch the upstream clone. Deliberately self-contained: it does not import
#    common/, so it fixes the repo even when the mounted dataset is older than
#    this notebook. Idempotent - safe to re-run.
#
#    - pad_center(fft_window, filter_length)  -> np.pad   (librosa >= 0.10: size is keyword-only)
#    - librosa_mel_fn(a, b, c, d, e)          -> sr=/n_fft=/n_mels=/fmin=/fmax=
#    - hard-coded .cuda()                     -> device-agnostic tensors
#    - short-clip guards + UTF-8 .lab read in preprocessor.py
#    - torch.load(..., weights_only=False)    (torch >= 2.6)
#    - Telugu: "@<grapheme>" symbols from data/vocab.json
import json, re, subprocess, sys
from pathlib import Path

repo = Path(f"{SSTF}/third_party/FastSpeech2_{LANG}")
assert (repo / "train.py").exists(), f"repo missing: {repo} - run notebook 01 first"

# --- audio/stft.py -----------------------------------------------------------
stft = repo / "audio" / "stft.py"
src = stft.read_text(encoding="utf-8")
src = re.sub(
    r"fft_window = pad_center\(fft_window,\s*filter_length\)",
    "fft_window = np.pad(fft_window, ((filter_length - len(fft_window)) // 2, "
    "filter_length - len(fft_window) - (filter_length - len(fft_window)) // 2))",
    src)
src = re.sub(
    r"librosa_mel_fn\(\s*sampling_rate,\s*filter_length,\s*n_mel_channels,"
    r"\s*mel_fmin,\s*mel_fmax,?\s*\)",
    "librosa_mel_fn(sr=sampling_rate, n_fft=filter_length, n_mels=n_mel_channels, "
    "fmin=mel_fmin, fmax=mel_fmax)", src)
src = src.replace("input_data.cuda(),", "input_data.to(self.forward_basis.device),")
src = src.replace(
    "torch.autograd.Variable(self.forward_basis, requires_grad=False).cuda(),",
    "self.forward_basis.to(input_data.device),")
src = src.replace(
    "torch.autograd.Variable(self.inverse_basis, requires_grad=False),",
    "self.inverse_basis.to(recombine_magnitude_phase.device),")
src = src.replace(
    "window_sum = window_sum.cuda() if magnitude.is_cuda else window_sum",
    "window_sum = window_sum.to(magnitude.device)")
stft.write_text(src, encoding="utf-8")

# --- audio/audio_processing.py: the same pad_center call ---------------------
ap = repo / "audio" / "audio_processing.py"
src = ap.read_text(encoding="utf-8")
src = re.sub(
    r"win_sq = librosa_util\.pad_center\(win_sq,\s*n_fft\)",
    "win_sq = np.pad(win_sq, ((n_fft - len(win_sq)) // 2, "
    "n_fft - len(win_sq) - (n_fft - len(win_sq)) // 2))",
    src)
ap.write_text(src, encoding="utf-8")

# --- preprocessor/preprocessor.py: short clips + UTF-8 labels ----------------
pp = repo / "preprocessor" / "preprocessor.py"
src = pp.read_text(encoding="utf-8")
src = src.replace('with open(text_path, "r") as f:',
                  'with open(text_path, "r", encoding="utf-8") as f:')
for var in ("pitch", "energy"):
    old = ("            for i, d in enumerate(duration):\n"
           "                if d > 0:\n"
           f"                    {var}[i] = np.mean({var}[pos : pos + d])")
    new = ("            for i, d in enumerate(duration):\n"
           f"                if i >= len({var}):\n"
           "                    break\n"
           "                if d > 0:\n"
           f"                    end = min(pos + d, len({var}))\n"
           "                    if end > pos:\n"
           f"                        {var}[i] = np.mean({var}[pos:end])\n"
           "                    else:\n"
           f"                        {var}[i] = 0")
    src = src.replace(old, new)
pp.write_text(src, encoding="utf-8")

# --- utils/model.py: torch >= 2.6 -------------------------------------------
mp = repo / "utils" / "model.py"
src = mp.read_text(encoding="utf-8")
src = src.replace("ckpt = torch.load(ckpt_path)",
                  "ckpt = torch.load(ckpt_path, weights_only=False)")
src = src.replace('torch.load("hifigan/generator_LJSpeech.pth.tar")',
                  'torch.load("hifigan/generator_LJSpeech.pth.tar", weights_only=False)')
src = src.replace('torch.load("hifigan/generator_universal.pth.tar")',
                  'torch.load("hifigan/generator_universal.pth.tar", weights_only=False)')
mp.write_text(src, encoding="utf-8")

# --- Telugu: "@<grapheme>" symbols ------------------------------------------
if LANG == "Telugu":
    vocab = json.loads(Path("project/Telugu/data/vocab.json").read_text(encoding="utf-8"))
    units = [u for u in vocab if u not in ("_", "~")]
    sym = repo / "text" / "symbols.py"
    src = sym.read_text(encoding="utf-8")
    begin = "# --- SSTF PATCH BEGIN (telugu graphemes) ---"
    end = "# --- SSTF PATCH END (telugu graphemes) ---"
    if begin in src:
        head, rest = src.split(begin, 1)
        src = head + rest.split(end, 1)[1].lstrip("\n")
    sym.write_text(
        src.rstrip("\n") + "\n\n" + begin + "\n"
        f"_telugu_units = {json.dumps(units, ensure_ascii=False)}\n"
        "_telugu_symbols = ['@' + u for u in _telugu_units]\n"
        "symbols = symbols + [s for s in _telugu_symbols if s not in symbols]\n"
        + end + "\n", encoding="utf-8")
    print("Telugu grapheme symbols appended:", len(units))

# --- verify: the exact call that failed, plus the inverse path --------------
check = """
import torch
from audio.stft import TacotronSTFT
s = TacotronSTFT(1024, 256, 1024, 80, 22050, 0.0, 8000.0)
y = (torch.rand(1, 16000) - 0.5) * 1.8
m, e = s.mel_spectrogram(y)
r = s.stft_fn.inverse(*s.stft_fn.transform(y))
print('PATCHED OK  mel', tuple(m.shape), '| inverse', tuple(r.shape))
"""
r = subprocess.run([sys.executable, "-c", check], cwd=repo, capture_output=True, text=True)
print((r.stdout.strip() or r.stderr[-2000:]))
assert r.returncode == 0, "audio stack still broken - paste the traceback above"


In [ ]:
# 4) Smoke-test the MMS_FA alignment on 20 utterances (~1 min).
#    Inspect one TextGrid afterwards: the 'phones' tier must hold phones for
#    English and grapheme clusters for Telugu.
!python project/{LANG}/preprocess.py --stage align --limit 20


In [ ]:
# 5) FULL alignment (both splits). 5 h of 22 kHz audio ~ 1.5-2.5 h on T4.
#    Safe to re-run: existing TextGrids are skipped.
!python project/{LANG}/preprocess.py --stage align


In [ ]:
# 6) Feature extraction: stages wav/.lab/TextGrid into the repo layout, calls
#    the upstream preprocess.py (mel, pitch, energy, duration), then rewrites
#    train.txt / val.txt from our recorded, disjoint split.
!python project/{LANG}/preprocess.py --stage repo


In [ ]:
# 7) Sanity: what the model will actually train on.
#    Self-contained: it measures the features on disk, so it does not care
#    which build wrote _stage_complete.json.
import json, sys
from pathlib import Path
import numpy as np

sys.path.insert(0, SSTF)
from common import config

cfg = config.load(LANG)
stats = json.load(open(f"project/{LANG}/data/split_stats.json"))
repo = Path(config.require_repo(LANG))
pre = repo / "preprocessed_data" / cfg["dataset"]["name"]
marker = pre / "_stage_complete.json"
assert marker.exists(), f"feature extraction did not finish ({marker}) - run cell 6 first"

info = json.loads(marker.read_text(encoding="utf-8"))
hop, sr = cfg["audio"]["hop_length"], cfg["audio"]["sampling_rate"]

missing = [d for d in ("mel", "pitch", "energy", "duration") if not (pre / d).is_dir()]
assert not missing, f"missing feature directories: {missing}"

# duration holds one frame count PER PHONEME, so the number of frames is
# .sum() - len() would be the phoneme count and under-report the hours by ~10x.
measured = {}
for split in ("train", "val"):
    rows = [l.split("|") for l in
            (pre / f"{split}.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
    frames = sum(int(np.load(pre / "duration" / f"{spk}-duration-{name}.npy").sum())
                 for name, spk, *_ in rows)
    measured[split] = (len(rows), frames * hop / sr / 3600)
    print(f"{split}: {len(rows)} utts, {measured[split][1]:.2f} h")

kept = sum(n for n, _ in measured.values())
dropped = info["staged"] - kept
print(f"staged {info['staged']} | kept {kept} | removed by preprocessor {dropped}")

# Upstream drops utterances with an empty alignment or no voiced pitch.
# A handful out of thousands is expected; >1% means the alignment is broken.
assert dropped >= 0, "more utterances were kept than staged - stale marker, re-run cell 6"
assert dropped <= 0.01 * info["staged"], \
    "more than 1% of utterances failed preprocessing - check cell 5 (alignment)"

# measured vs planned, then the assignment's 5 h / 1 h budget
# Use split_stats.json as ground truth (proportional split when data is limited)
assert abs(measured["train"][1] - stats["train_hours"]) <= 0.1, \
    (round(measured["train"][1], 3), stats["train_hours"])
assert abs(measured["val"][1] - stats["val_hours"]) <= 0.1, \
    (round(measured["val"][1], 3), stats["val_hours"])
# Sanity: both splits non-empty and disjoint
assert measured["train"][1] > 0, measured["train"][1]
assert measured["val"][1] > 0, measured["val"][1]

print(f"OK: {measured['train'][1]:.2f} h train / {measured['val'][1]:.2f} h val, "
      f"disjoint split, {dropped} dropped (<1%)")
